In [ ]:
import os
import re

import numpy as np
import pandas as pd


In [ ]:
# Runtime paths (edit before running)
from pathlib import Path

# MIMIC-IV core tables (e.g., admissions.csv under hosp/)
path_to_mimic = "/path/to/mimiciv/3.0"

# MIMIC-IV-ED tables
path_to_mimic_ed = "/path/to/mimiciv-ed"

# MIMIC-IV-ECG waveform root
path_to_mimic_ecg = "/path/to/mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0"

# Output directory for generated files
path_to_local_files = str(Path("../../").resolve())


In [ ]:
admissions = pd.read_csv(os.path.join(path_to_mimic, "hosp/admissions.csv"))
ed_stays = pd.read_csv(os.path.join(path_to_mimic_ed, "edstays.csv"))

print("admissions:", admissions.shape)
print("ed_stays:", ed_stays.shape)


In [ ]:
ecg_dataset = pd.read_csv(mimic_ecg_dataset_csv)
print("ecg_dataset:", ecg_dataset.shape)
ecg_dataset.columns


In [ ]:
_ecg_sim = ecg_dataset[["subject_id", "study_id", "ecg_time", "edregtime", "ed_stay_id", "hadm_id"]].copy()

_ad_dea = pd.merge(
    _ecg_sim,
    admissions[["hadm_id", "hospital_expire_flag", "admittime", "deathtime"]],
    on="hadm_id",
    how="left",
)

_ad_ed_dea = pd.merge(
    _ad_dea,
    ed_stays[["stay_id", "intime", "outtime", "disposition"]],
    left_on="ed_stay_id",
    right_on="stay_id",
    how="left",
)

# 3.4 ED-expired
_ad_ed_dea["ed_expired"] = (_ad_ed_dea["disposition"] == "EXPIRED").astype(int)

# 3.5 ED length (hours) for EXPIRED
_ed_expired = _ad_ed_dea[_ad_ed_dea["disposition"] == "EXPIRED"].copy()
_ed_expired["ed_length_hours"] = (
    pd.to_datetime(_ed_expired["outtime"], dayfirst=True, errors="coerce")
    - pd.to_datetime(_ed_expired["intime"], dayfirst=True, errors="coerce")
).dt.total_seconds() / 3600

_ad_ed_dea = pd.merge(
    _ad_ed_dea,
    _ed_expired[["ed_stay_id", "ed_length_hours"]],
    on="ed_stay_id",
    how="left",
)

# 3.6 ED -> death time delta (hours)
_ad_ed_dea["deathtime"] = pd.to_datetime(_ad_ed_dea["deathtime"], dayfirst=True, errors="coerce")
_ad_ed_dea["edregtime"] = pd.to_datetime(_ad_ed_dea["edregtime"], errors="coerce")

_ad_ed_dea["ed_todeath"] = _ad_ed_dea["deathtime"] - _ad_ed_dea["edregtime"]
_ad_ed_dea["ed_todeath_hours"] = _ad_ed_dea["ed_todeath"].dt.total_seconds() / 3600

# 3.7 death: in-hospital OR ED-expired
_ad_ed_dea["death"] = (
    (_ad_ed_dea["hospital_expire_flag"] == 1) | (_ad_ed_dea["ed_expired"] == 1)
).astype(int)

# 3.8 window labels
for w in [24, 48, 72]:
    _ad_ed_dea[f"death_{w}h"] = 0

_mask = (_ad_ed_dea["ed_expired"] == 1) & (_ad_ed_dea["hospital_expire_flag"] == 0)
for w in [24, 48, 72]:
    _ad_ed_dea.loc[_mask & (_ad_ed_dea["ed_length_hours"] <= w), f"death_{w}h"] = 1
    _ad_ed_dea[f"death_{w}h"] = (
        (_ad_ed_dea[f"death_{w}h"] == 1) | (_ad_ed_dea["ed_todeath_hours"] <= w)
    ).astype(int)

_death_cols = ["study_id", "death", "ed_todeath_hours", "death_24h", "death_48h", "death_72h"]
ecg_dataset = pd.merge(ecg_dataset, _ad_ed_dea[_death_cols], on="study_id", how="left")

print(ecg_dataset[["death", "death_24h", "death_48h", "death_72h"]].sum(numeric_only=True))


In [ ]:
triage = pd.read_csv(os.path.join(path_to_mimic_ed, "triage.csv"))

triage_cols = [
    "stay_id",
    "temperature",
    "resprate",
    "o2sat",
    "sbp",
    "pain",
    "acuity",
]

_ecg_multi = pd.merge(
    ecg_dataset,
    triage[triage_cols],
    left_on="ed_stay_id",
    right_on="stay_id",
    how="left",
)

print(_ecg_multi[triage_cols[1:]].isna().sum())
print("ecg_multi:", _ecg_multi.shape)

_ecg_multi.to_csv(ecg_multi_csv, index=False)
ecg_multi_csv


In [ ]:
df = _ecg_multi.copy()

def _detect_temp_unit(s: pd.Series) -> str:
    s_num = pd.to_numeric(s, errors="coerce")
    return "F" if s_num.dropna().median() > 45 else "C"

def _to_celsius(s: pd.Series, unit: str) -> pd.Series:
    s_num = pd.to_numeric(s, errors="coerce")
    return (s_num - 32) * 5.0 / 9.0 if unit == "F" else s_num

def clean_vitals_like_mdsed(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["temperature", "resprate", "o2sat", "sbp", "dbp"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "temperature" in df.columns:
        unit = _detect_temp_unit(df["temperature"])
        df["temperature_c"] = _to_celsius(df["temperature"], unit)
        df.loc[(df["temperature_c"] < 10) | (df["temperature_c"] > 65), "temperature_c"] = np.nan

    if "resprate" in df.columns:
        df.loc[(df["resprate"] > 300) | (df["resprate"] <= 0), "resprate"] = np.nan
    if "o2sat" in df.columns:
        df.loc[(df["o2sat"] < 0) | (df["o2sat"] > 100), "o2sat"] = np.nan
    if "sbp" in df.columns:
        df.loc[(df["sbp"] > 500) | (df["sbp"] <= 0), "sbp"] = np.nan
    if "dbp" in df.columns:
        df.loc[(df["dbp"] > 500) | (df["dbp"] <= 0), "dbp"] = np.nan
    return df

def clean_demographics_like_mdsed(df: pd.DataFrame) -> pd.DataFrame:
    if "gender" in df.columns:
        df["gender_clean"] = df["gender"].map({"M": 1, "F": 0})
        df["gender_unknown"] = df["gender_clean"].isna().astype(int)

    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan
    elif "anchor_age" in df.columns:
        df["age"] = pd.to_numeric(df["anchor_age"], errors="coerce")
        df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan

    return df

df = clean_vitals_like_mdsed(df)
df = clean_demographics_like_mdsed(df)

df[["temperature", "temperature_c", "resprate", "o2sat", "sbp", "age"]].describe(include="all")


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def add_general_strat_fold(
    df: pd.DataFrame,
    label_col: str = "death_72h",
    group_col: str = "subject_id",
    n_splits: int = 10,
    random_state: int = 23,
) -> pd.DataFrame:
    out = df.copy()
    y = out[label_col].fillna(0).astype(int).values
    groups = out[group_col].values

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    out["general_strat_fold"] = -1
    for fold, (_, te_idx) in enumerate(sgkf.split(out, y, groups)):
        out.loc[out.index[te_idx], "general_strat_fold"] = fold

    return out

df = add_general_strat_fold(df, label_col="death_72h", group_col="subject_id", n_splits=10, random_state=23)
print(df.groupby("general_strat_fold")["death_72h"].mean())


In [ ]:
df.columns

## Main-Cohort Dataset Construction

This section constructs the main cohort dataset and prepares tabular-waveform inputs for model training.


In [ ]:
import wfdb

TARGET_FS = 100
TARGET_LEN = 1000
N_CHANNELS = 12

def _resample_to(x_tc: np.ndarray, fs_from: int, fs_to: int) -> np.ndarray:
    if fs_from == fs_to:
        return x_tc
    import scipy.signal
    T, C = x_tc.shape
    n = int(round(T * fs_to / fs_from))
    return scipy.signal.resample(x_tc, n, axis=0)

def _pad_or_crop_center(x_ct: np.ndarray, target_len: int) -> np.ndarray:
    C, T = x_ct.shape
    if T == target_len:
        return x_ct
    if T > target_len:
        start = (T - target_len) // 2
        return x_ct[:, start : start + target_len]
    out = np.zeros((C, target_len), dtype=x_ct.dtype)
    out[:, :T] = x_ct
    return out

def load_one_ecg(path: str, *, target_fs: int = TARGET_FS, target_len: int = TARGET_LEN, clip_amp: float = 3.0) -> np.ndarray:
    rec = wfdb.rdrecord(path)
    x = rec.p_signal
    fs = int(round(rec.fs))

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    x = np.clip(x, -clip_amp, clip_amp)

    x = _resample_to(x, fs_from=fs, fs_to=target_fs)  # (T, C)
    x = x.T  # (C, T)
    x = _pad_or_crop_center(x, target_len=target_len)
    if x.shape[0] != N_CHANNELS:
        raise ValueError(f"unexpected channels: {x.shape}")
    return x

def build_dataset_array(df: pd.DataFrame, *, pre_path: str, path_col: str = "path", target_fs: int = TARGET_FS, target_len: int = TARGET_LEN) -> np.ndarray:
    k = len(df)
    X = np.zeros((k, N_CHANNELS, target_len), dtype=np.float32)

    for i, rel in enumerate(df[path_col].astype(str).values):
        if i % 1000 == 0 and i > 0:
            print(f"[load] {i}/{k}")
        rec_path = os.path.join(pre_path, rel)
        X[i] = load_one_ecg(rec_path, target_fs=target_fs, target_len=target_len)

    return X

pre_path = path_to_mimic_ecg
X = build_dataset_array(df=df, pre_path=pre_path, path_col="path")
X.shape


In [ ]:
x_train = df[df["general_strat_fold"].isin(range(0, 7))].reset_index(drop=True)
x_val = df[df["general_strat_fold"].isin([7])].reset_index(drop=True)
x_test = df[df["general_strat_fold"].isin(range(8, 10))].reset_index(drop=True)

y_train_24_death = x_train["death_24h"].fillna(0).astype(int).values
y_val_24_death = x_val["death_24h"].fillna(0).astype(int).values
y_test_24_death = x_test["death_24h"].fillna(0).astype(int).values

y_train_48_death = x_train["death_48h"].fillna(0).astype(int).values
y_val_48_death = x_val["death_48h"].fillna(0).astype(int).values
y_test_48_death = x_test["death_48h"].fillna(0).astype(int).values

y_train_72_death = x_train["death_72h"].fillna(0).astype(int).values
y_val_72_death = x_val["death_72h"].fillna(0).astype(int).values
y_test_72_death = x_test["death_72h"].fillna(0).astype(int).values

print("Train/Val/Test sizes:", len(x_train), len(x_val), len(x_test))
print("72h positives:", (y_train_72_death==1).sum(), (y_val_72_death==1).sum(), (y_test_72_death==1).sum())


In [ ]:
features = ["temperature_c", "resprate", "o2sat", "sbp", "age", "gender_clean"]

for col in features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = np.nan

features_with_masks = []
for col in features:
    mask_col = col + "_m"
    df[mask_col] = df[col].notna().astype(float)
    features_with_masks.extend([col, mask_col])

selected_folds = df[df["general_strat_fold"].isin(range(0, 7))]
medians = selected_folds[features].median(numeric_only=True)
df[features] = df[features].fillna(medians)

x_train_tab = (
    df[df["general_strat_fold"].isin(range(0, 7))][features_with_masks]
    .reset_index(drop=True)
    .to_numpy(dtype=np.float32)
)
x_val_tab = (
    df[df["general_strat_fold"].isin([7])][features_with_masks]
    .reset_index(drop=True)
    .to_numpy(dtype=np.float32)
)
x_test_tab = (
    df[df["general_strat_fold"].isin(range(8, 10))][features_with_masks]
    .reset_index(drop=True)
    .to_numpy(dtype=np.float32)
)

np.savez(
    train_npz,
    X,
    x_train=x_train_tab,
    x_val=x_val_tab,
    x_test=x_test_tab,
    y_train_24_death=y_train_24_death,
    y_val_24_death=y_val_24_death,
    y_test_24_death=y_test_24_death,
    y_train_48_death=y_train_48_death,
    y_val_48_death=y_val_48_death,
    y_test_48_death=y_test_48_death,
    y_train_72_death=y_train_72_death,
    y_val_72_death=y_val_72_death,
    y_test_72_death=y_test_72_death,
    subject_id=df["subject_id"].values,
    file_name=df["file_name"].values,
)
train_npz
